# PathSnap — Notebook 06: Bayesian hyperparameter optimisation of hist_gb (Milestone 6)
**Owner: C (nisarg-ravi)** · model selected in M5 (D21): `HistGradientBoostingClassifier`.

* **Data:** 80 training trips only (5-fold GroupKFold by trip, trips 10+16 grouped, the *same folds as M4*).
  Student-test is used once at the end to compare tuned vs untuned. Evaluator trips are never touched.
* **Strategy:** Bayesian optimisation with a Gaussian-process surrogate (Matérn-5/2, automatic relevance determination)
  and the Expected-Improvement acquisition (`pathsnap/bayesopt.py`). 10 quasi-random (Sobol) starting points, then each
  new trial is the configuration the GP expects to improve most on the best so far. Implemented with scikit-learn so the
  run is identical on Colab and locally (no extra package); every trial is logged and the run can resume.
* **Objective (maximise):** mean point-level Top-1 over the 5 validation folds, the primary metric (D19).
  Log loss and macro F1 are logged for every trial so we can check that probability quality doesn't get worse.

In [ ]:
import os, sys, json, time, zipfile
import numpy as np, pandas as pd, matplotlib.pyplot as plt
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive/PathSnap"
    ZIP_PATH, EXTRACT_DIR = f"{DRIVE}/Student folder.zip", "/content/pathsnap_data"
    DS_DIR, M4, OUT = f"{DRIVE}/m3_outputs/dataset", f"{DRIVE}/m4_outputs", f"{DRIVE}/m6_outputs"
    REPO = "/content/01_pathsnap"
    if not os.path.isdir(REPO): os.system(f"git clone -q https://github.com/paya5am/01_pathsnap {REPO}")
    PKG_PARENT = REPO if os.path.isdir(f"{REPO}/pathsnap") else DRIVE
else:
    ZIP_PATH = os.environ.get("PATHSNAP_ZIP", "Student folder.zip"); EXTRACT_DIR = os.environ.get("PATHSNAP_EXTRACT", "pathsnap_data")
    DS_DIR = os.environ.get("PATHSNAP_DS", "m3_outputs/dataset"); M4 = os.environ.get("PATHSNAP_M4", "m4_outputs")
    OUT = os.environ.get("PATHSNAP_M6", "m6_outputs"); PKG_PARENT = os.environ.get("PATHSNAP_PKG", ".")
sys.path.insert(0, PKG_PARENT)
if not os.path.isdir(EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(EXTRACT_DIR)
DATA_ROOT = next(dp for dp, dn, _ in os.walk(EXTRACT_DIR) if "train" in dn and "student_test" in dn)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, log_loss
from pathsnap.modeling import load_dataset, arc_tables, run_experiment, point_predictions
from pathsnap.bayesopt import Space, optimise
from pathsnap.features import FEATURES
from pathsnap.config import SEED
os.makedirs(OUT, exist_ok=True)
try: display
except NameError: display = print
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
F, P = load_dataset(DS_DIR)
TR = F[F.partition == "train"].reset_index(drop=True)
FOLDS = list(GroupKFold(n_splits=5).split(TR, groups=TR.cv_group.values))     # identical to M4's split
print("train rows", len(TR), "| folds:", [len(v) for _, v in FOLDS])

## 1. Search space (and why)
| Hyperparameter | M4 value | Range searched | Scale | Why this range |
|---|---|---|---|---|
| `learning_rate` | 0.1 | 0.02 – 0.3 | log | step size of each boosting round; smaller = more rounds, usually better generalisation; above 0.3 is unstable |
| `max_leaf_nodes` | 31 | 15 – 255 | log, int | tree size = order of feature interactions; 16 features with strong interactions (heading × moving) may need bigger trees |
| `min_samples_leaf` | 20 | 10 – 500 | log, int | ~740k training rows per fold: leaves of 20 may memorise single GPS points; up to 500 regularises |
| `l2_regularization` | 0 | 1e-4 – 10 | log | shrinks leaf values; helps calibration (log loss → HMM emissions) |
| `max_features` | 1.0 | 0.4 – 1.0 | linear | share of features tried per split; < 1 decorrelates trees (needs scikit-learn ≥ 1.4) |

Fixed: `max_iter` = 600 with early stopping (10% internal split *inside the training folds*, patience 20), so the
number of trees adapts to the learning rate; `max_bins` = 255; `random_state` = 42.

In [ ]:
SPACE = Space([("learning_rate", 0.02, 0.3, "log"), ("max_leaf_nodes", 15, 255, "logint"),
               ("min_samples_leaf", 10, 500, "logint"), ("l2_regularization", 1e-4, 10.0, "log"),
               ("max_features", 0.4, 1.0, "lin")])
FIXED = dict(max_iter=600, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, max_bins=255, random_state=SEED)

def make_hgb(p):
    return HistGradientBoostingClassifier(**p, **FIXED)

def cv_objective(params):
    top1, ll, mf1, iters = [], [], [], []
    for tr_idx, va_idx in FOLDS:
        dtr = TR.iloc[tr_idx]; dtr = dtr[dtr.label >= 0]; dva = TR.iloc[va_idx].reset_index(drop=True)
        m = make_hgb(params).fit(dtr[FEATURES].values, dtr.label.values)
        p = m.predict_proba(dva[FEATURES].values)[:, 1]
        pp = point_predictions(dva, p); lab_pts = pp[pp.label_status == 0]
        n_lab = (P[(P.partition == "train") & P.trip_id.isin(dva.trip_id.unique())].label_status == 0).sum()
        top1.append(lab_pts.pred_is_ref.sum() / n_lab)
        l = dva.label.values >= 0; y = dva.label.values[l]; q = np.clip(p[l], 1e-7, 1 - 1e-7)
        ll.append(log_loss(y, q, labels=[0, 1])); mf1.append(f1_score(y, (q >= .5).astype(int), average="macro")); iters.append(m.n_iter_)
    return {"score": float(np.mean(top1)), "top1_std": float(np.std(top1, ddof=1)), "log_loss": float(np.mean(ll)),
            "macro_f1": float(np.mean(mf1)), "mean_trees": float(np.mean(iters)), "fold_top1": json.dumps([round(x, 5) for x in top1])}

## 2. Reference point: the untuned (M4) configuration under the same objective

In [ ]:
BASE = {"learning_rate": 0.1, "max_leaf_nodes": 31, "min_samples_leaf": 20, "l2_regularization": 0.0, "max_features": 1.0}
base_path = f"{OUT}/untuned_cv.json"
if os.path.isfile(base_path):
    base_res = json.load(open(base_path))
else:
    t0 = time.time(); base_res = cv_objective(BASE); base_res["seconds"] = time.time() - t0
    json.dump(base_res, open(base_path, "w"), indent=1)
print("untuned (M4 config, max_iter 600):", {k: (round(v, 4) if isinstance(v, float) else v) for k, v in base_res.items()})

## 3. Run the optimisation (30 trials: 10 Sobol + 20 GP-EI) — resumes if interrupted

In [ ]:
N_TRIALS, N_INIT = 30, 10   # stopped at 30: no improvement after trial 8 (see convergence plot); time budget
t0 = time.time()
TRIALS = optimise(cv_objective, SPACE, n_trials=N_TRIALS, n_init=N_INIT, log_path=f"{OUT}/bo_trials.csv", seed=SEED)
print(f"optimisation wall time this session: {time.time()-t0:.0f}s | total trial time {TRIALS.seconds.sum():.0f}s")

## 4. Results of the search

In [ ]:
TRIALS = pd.read_csv(f"{OUT}/bo_trials.csv")
best = TRIALS.loc[TRIALS.score.idxmax()]
pnames = [d[0] for d in SPACE.dims]
BEST = {k: (int(best[k]) if k in ("max_leaf_nodes", "min_samples_leaf") else float(best[k])) for k in pnames}
print("best trial:", int(best.trial), "| CV Top-1 %.4f ± %.4f | log loss %.4f | macro F1 %.4f | mean trees %.0f" %
      (best.score, best.top1_std, best.log_loss, best.macro_f1, best.mean_trees))
print("best hyperparameters:", BEST)
print("untuned: CV Top-1 %.4f ± %.4f | log loss %.4f" % (base_res["score"], base_res["top1_std"], base_res["log_loss"]))
top = TRIALS.sort_values("score", ascending=False).head(10)
display(top[["trial", "how"] + pnames + ["score", "top1_std", "log_loss", "macro_f1", "mean_trees", "seconds"]].round(4))
# paired per-fold gain of the best config over the untuned config (same folds)
bf = np.array(json.loads(best.fold_top1)); uf = np.array(json.loads(base_res["fold_top1"]))
print("per-fold Top-1 gain (tuned − untuned):", np.round(bf - uf, 4).tolist(), "| folds improved:", int((bf > uf).sum()), "/ 5")
json.dump({"best_params": BEST, "fixed": {k: v for k, v in FIXED.items()}, "best_trial": int(best.trial),
           "cv_top1": float(best.score), "cv_top1_std": float(best.top1_std), "cv_log_loss": float(best.log_loss),
           "untuned_cv_top1": base_res["score"], "per_fold_gain": (bf - uf).tolist(),
           "n_trials": int(len(TRIALS)), "total_trial_seconds": float(TRIALS.seconds.sum())},
          open(f"{OUT}/best_params.json", "w"), indent=1)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(TRIALS.trial, TRIALS.score, "o", color="0.6", label="trial")
ax[0].plot(TRIALS.trial, TRIALS.score.cummax(), "r-", lw=2, label="best so far")
ax[0].axhline(base_res["score"], ls="--", c="k", label=f"untuned {base_res['score']:.4f}")
ax[0].axvline(N_INIT - .5, c="b", ls=":", label="Sobol → GP-EI")
ax[0].set(xlabel="trial", ylabel="CV Top-1 (5 trip-grouped folds)", title="Bayesian optimisation convergence"); ax[0].legend(fontsize=8)
ax[1].scatter(TRIALS.log_loss, TRIALS.score, c=TRIALS.trial, cmap="viridis"); ax[1].scatter([base_res["log_loss"]], [base_res["score"]], marker="*", s=200, c="k", label="untuned")
ax[1].set(xlabel="CV log loss", ylabel="CV Top-1", title="Accuracy vs probability quality per trial (colour = trial)"); ax[1].legend()
plt.tight_layout(); plt.savefig(f"{OUT}/01_bo_convergence.png", dpi=130); plt.show()
fig, axs = plt.subplots(1, len(pnames), figsize=(4 * len(pnames), 3.5))
for a, p in zip(axs, pnames):
    a.scatter(TRIALS[p], TRIALS.score, c=TRIALS.trial, cmap="viridis", s=20); a.axvline(BEST[p], c="r", lw=1)
    if p in ("learning_rate", "max_leaf_nodes", "min_samples_leaf", "l2_regularization"): a.set_xscale("log")
    a.set(xlabel=p, ylabel="CV Top-1" if p == pnames[0] else "")
plt.suptitle("Each hyperparameter vs CV Top-1 (red = best)"); plt.tight_layout(); plt.savefig(f"{OUT}/02_bo_params.png", dpi=130); plt.show()

## 5. Refit the tuned model on all 80 training trips and compare tuned vs untuned
Same harness, metrics and student-test evaluation as M4 (`run_experiment`), so the comparison is like for like.

In [ ]:
ARCS = arc_tables(DATA_ROOT, [tuple(x) for x in F[["partition", "trip_id"]].drop_duplicates().itertuples(index=False)])
cfg = {**BEST, **{k: v for k, v in FIXED.items() if k != "random_state"}}
res_t, model_t = run_experiment("hist_gb_tuned", "Boosting", lambda: make_hgb(BEST), F, P, ARCS, cfg, OUT,
                                history_fn=lambda m: {"train_neg_log_loss": list(map(float, m.train_score_)),
                                                      "val_neg_log_loss": list(map(float, m.validation_score_))})
res_u = json.load(open(f"{M4}/hist_gb_results.json"))
def row(r, name):
    return {"model": name, "CV Top-1": r["cv"]["val_top1_accuracy"]["mean"], "CV Top-1 std": r["cv"]["val_top1_accuracy"]["std"],
            "CV macro F1": r["cv"]["val_row_macro_f1"]["mean"], "CV log loss": r["cv"]["val_log_loss"]["mean"],
            "test Top-1": r["student_test"]["top1_accuracy"], "test macro F1": r["student_test"]["row_macro_f1"],
            "test log loss": r["student_test"]["log_loss"], "test continuity": r["student_test"]["route_continuity"],
            "test route-arc F1": r["student_test"]["route_arc_f1"], "train-fold Top-1": r["cv"]["train_top1"]["mean"],
            "train time s": r["train_time_s"], "inference ms/1000 pts": r["inference_ms_per_1000_points"]}
CMP = pd.DataFrame([row(res_u, "hist_gb untuned (M4)"), row(res_t, "hist_gb tuned (M6)")]).set_index("model")
CMP.loc["change"] = CMP.iloc[1] - CMP.iloc[0]
CMP.to_csv(f"{OUT}/03_tuned_vs_untuned.csv"); display(CMP.T.round(4))
print(f"Computational cost: {len(TRIALS)} trials × 5 folds = {5*len(TRIALS)} model fits, {TRIALS.seconds.sum()/60:.1f} min of trial time "
      f"(+ untuned reference {base_res.get('seconds', 0):.0f}s).")
fig, ax = plt.subplots(figsize=(9, 4))
cols = ["CV Top-1", "CV macro F1", "test Top-1", "test macro F1"]
x = np.arange(len(cols)); ax.bar(x - .2, CMP.loc["hist_gb untuned (M4)", cols], .4, label="untuned (M4)")
ax.bar(x + .2, CMP.loc["hist_gb tuned (M6)", cols], .4, label="tuned (M6)"); ax.set_xticks(x); ax.set_xticklabels(cols)
ax.set(ylim=(0.8, 0.95), title="hist_gb: untuned vs Bayesian-tuned (CV = 80 train trips; test = 10 student-test trips)"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT}/03_tuned_vs_untuned.png", dpi=130); plt.show()

## 6. Is the "best" trial really better? Second candidate for the HMM
The best trial beats the untuned model by only ~0.001 CV Top-1 (fold std ≈ 0.02) and has slightly *worse* log loss.
The maximum of 30 noisy evaluations is optimistic ("winner's curse"). The HMM (M7) consumes probabilities, so we also
keep the configuration with the **lowest log loss among trials whose CV Top-1 is within 0.002 of the best** (a
tolerance ~1/10 of the fold std). M7 then decides between untuned / best-Top-1 / best-log-loss by the quantity that
matters for the system: Top-1 **after** HMM decoding, on the training trips (out-of-fold).

In [ ]:
TIE_TOL = 0.002
tied = TRIALS[TRIALS.score >= TRIALS.score.max() - TIE_TOL]
best_ll = tied.loc[tied.log_loss.idxmin()]
BEST_LL = {k: (int(best_ll[k]) if k in ("max_leaf_nodes", "min_samples_leaf") else float(best_ll[k])) for k in pnames}
print(f"tied trials (Top-1 ≥ {TRIALS.score.max() - TIE_TOL:.4f}): {tied.trial.tolist()}")
print(f"lowest log loss among them: trial {int(best_ll.trial)} | CV Top-1 {best_ll.score:.4f} | log loss {best_ll.log_loss:.4f} | trees {best_ll.mean_trees:.0f}")
print("params:", BEST_LL)
res_ll, _ = run_experiment("hist_gb_tuned_ll", "Boosting", lambda: make_hgb(BEST_LL), F, P, ARCS,
                           {**BEST_LL, **{k: v for k, v in FIXED.items() if k != "random_state"}}, OUT)
CMP2 = pd.DataFrame([row(res_u, "untuned (M4)"), row(res_t, f"best Top-1 (trial {int(best.trial)})"),
                     row(res_ll, f"best log loss among tied (trial {int(best_ll.trial)})")]).set_index("model")
CMP2.to_csv(f"{OUT}/04_three_candidates.csv"); display(CMP2.T.round(4))
bp = json.load(open(f"{OUT}/best_params.json")); bp.update({"best_logloss_tied_params": BEST_LL, "best_logloss_tied_trial": int(best_ll.trial),
                                                              "tie_tolerance": TIE_TOL})
json.dump(bp, open(f"{OUT}/best_params.json", "w"), indent=1)